# Brain Surgery — Bước 2 (E4B): QLoRA SFT -> ORPO & Export GGUF Modular

Notebook chạy trọn gói trên **Google Colab Pro (L4 24GB hoặc A100 40GB)** cho checkpoint **`haku-nekooo/gemma4-e4b-nuclear-pruned`**:
1. **Stage 2A (SFT)**: Đóng băng Vision/Audio encoders (`requires_grad=False`), train LoRA Attention/MLP (`lr=1.5e-4`) + `embed_tokens` / `embed_tokens_per_layer` (PLE) / `lm_head` (`embedding_lr=1.5e-5`), loss chỉ tính trên `assistant_spans`.
2. **Stage 2B (ORPO 1 epoch)**: Khóa cứng cả Vision/Audio và embeddings/PLE, tinh chỉnh `beta=0.1, lr=8e-6` trị 6 nhóm lỗi `nc-agent`.
3. **Bất biến SHA-256**: So khớp từng tensor Vision/Audio trước và sau mỗi lần merge để bảo đảm tương thích tuyệt đối với `--mmproj`.
4. **Export Modular GGUF**: Sinh `Q4_K_M` / `IQ4_XS` (có `imatrix`) + `mmproj` riêng và smoke test `--jinja --single-turn`.

## 1. Cấu hình Pipeline & Đồng bộ Repo

In [ ]:
from pathlib import Path

BASE_HF_REPO = "haku-nekooo/gemma4-e4b-nuclear-pruned"
TARGET_HF_REPO = "haku-nekooo/gemma4-e4b-nuclear-distilled"
GITHUB_REPO = "https://github.com/just-haku/brain-surgery.git"

WORK_DIR = Path("/content/work_step2")
REPO_DIR = Path("/content/brain-surgery")
LLAMA_CPP_DIR = Path("/content/llama.cpp")

RAW_JSONL_INPUTS = [
    str(REPO_DIR / "data" / "distill" / "pilot_sft_raw.jsonl"),
    str(REPO_DIR / "data" / "distill" / "pilot_orpo_raw.jsonl"),
]

RUN_SFT = True
RUN_ORPO = True
TRAIN_EMBEDDINGS = True
USE_IMATRIX = True
EXPORT_QUANTS = ["Q4_K_M", "IQ4_XS"]
PUSH_TO_HF = False

In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "transformers>=5.5.0",
        "peft>=0.15.0",
        "trl>=0.16.0",
        "datasets>=3.0.0",
        "bitsandbytes>=0.45.0",
        "accelerate>=1.3.0",
        "huggingface_hub>=0.28.0",
        "gguf",
    ],
    check=True,
)

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, str(REPO_DIR)], check=True)

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import torch

vram_gb = (
    torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if torch.cuda.is_available()
    else 0.0
)
USE_4BIT = vram_gb < 36.0
print(f"GPU VRAM: {vram_gb:.1f} GB -> use_4bit={USE_4BIT}")

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata

        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    from huggingface_hub import login

    login(token=hf_token, add_to_git_credential=False)

## 2. Kiểm duyệt & Biên dịch Dataset (`ingest`)

In [ ]:
import json
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from distill.validator import convert_and_validate_openai_jsonl

WORK_DIR.mkdir(parents=True, exist_ok=True)
BASE_CKPT_DIR = Path(snapshot_download(repo_id=BASE_HF_REPO, local_dir=str(WORK_DIR / "base_ckpt")))
tok = AutoTokenizer.from_pretrained(str(BASE_CKPT_DIR), use_fast=True)

COMPILED_DIR = WORK_DIR / "compiled_data"
SFT_JSONL = COMPILED_DIR / "sft_gemma4.jsonl"
ORPO_JSONL = COMPILED_DIR / "orpo_gemma4.jsonl"
REJECTED_JSONL = COMPILED_DIR / "rejected.jsonl"

ingest_report = convert_and_validate_openai_jsonl(
    raw_jsonl_paths=RAW_JSONL_INPUTS,
    out_sft_path=SFT_JSONL,
    out_orpo_path=ORPO_JSONL,
    out_rejected_path=REJECTED_JSONL,
    tokenizer=tok,
    max_tokens=8192,
)
print(json.dumps(ingest_report, indent=2, ensure_ascii=False))
assert ingest_report["kept_sft"] > 0 and ingest_report["rejected_count"] == 0

## 3. Stage 2A — QLoRA/BF16 SFT (Vision/Audio Frozen, Low-LR Embeddings/PLE)

In [ ]:
from train.train_sft import SFTConfig, run_sft_training

SFT_MERGED_DIR = WORK_DIR / "stage2a_sft_merged"
if RUN_SFT:
    sft_cfg = SFTConfig(
        base_model_id=str(BASE_CKPT_DIR),
        train_jsonl=str(SFT_JSONL),
        output_dir=str(WORK_DIR / "stage2a_sft_lora"),
        merged_dir=str(SFT_MERGED_DIR),
        use_4bit=USE_4BIT,
        train_embeddings=TRAIN_EMBEDDINGS,
    )
    sft_summary = run_sft_training(sft_cfg)
    print(json.dumps(sft_summary, indent=2, ensure_ascii=False))

## 4. Stage 2B — 1-Epoch ORPO Alignment (Embeddings/PLE + Vision/Audio Frozen)

In [ ]:
from train.train_orpo import ORPOConfig, run_orpo_training

FINAL_MERGED_DIR = WORK_DIR / "stage2b_orpo_merged"
if RUN_ORPO:
    orpo_cfg = ORPOConfig(
        sft_merged_dir=str(SFT_MERGED_DIR),
        orpo_jsonl=str(ORPO_JSONL),
        output_dir=str(WORK_DIR / "stage2b_orpo_lora"),
        merged_dir=str(FINAL_MERGED_DIR),
        use_4bit=USE_4BIT,
        beta=0.1,
        learning_rate=8e-6,
    )
    orpo_summary = run_orpo_training(orpo_cfg)
    print(json.dumps(orpo_summary, indent=2, ensure_ascii=False))
else:
    FINAL_MERGED_DIR = SFT_MERGED_DIR

## 5. Export GGUF Modular (`Q4_K_M` / `IQ4_XS` + `--mmproj`) & Smoke Check

In [ ]:
import os
from train.export_gguf import export_modular_gguf, write_imatrix_calib_corpus

if (LLAMA_CPP_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(LLAMA_CPP_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_CPP_DIR)], check=True)

subprocess.run(["cmake", "-B", str(LLAMA_CPP_DIR / "build"), "-S", str(LLAMA_CPP_DIR), "-DGGML_CUDA=OFF"], check=True)
subprocess.run(
    [
        "cmake",
        "--build",
        str(LLAMA_CPP_DIR / "build"),
        "--config",
        "Release",
        "--target",
        "llama-quantize",
        "llama-imatrix",
        "llama-cli",
        f"-j{os.cpu_count() or 4}",
    ],
    check=True,
)

GGUF_DIR = WORK_DIR / "gguf"
calib_txt = write_imatrix_calib_corpus(SFT_JSONL, WORK_DIR / "imatrix_calib.txt", max_samples=64) if USE_IMATRIX else None
export_manifest = export_modular_gguf(
    model_dir=FINAL_MERGED_DIR,
    out_dir=GGUF_DIR,
    llama_cpp_dir=LLAMA_CPP_DIR,
    model_prefix="gemma4-e4b-nc",
    quants=EXPORT_QUANTS,
    mmproj_outtype="bf16",
    imatrix_calib_file=calib_txt,
)
print(json.dumps(export_manifest, indent=2, ensure_ascii=False))

cli_bin = LLAMA_CPP_DIR / "build" / "bin" / "llama-cli"
cli_help = subprocess.run([str(cli_bin), "--help"], capture_output=True, text=True).stdout
cli_st = ["--single-turn"] if "--single-turn" in cli_help else (["-no-cnv"] if "-no-cnv" in cli_help else [])
q4_path = export_manifest["artifacts"][EXPORT_QUANTS[0].upper()]["path"]
smoke = subprocess.run(
    [str(cli_bin), "-m", q4_path, "--jinja", "-p", "Viết 1 hàm Rust tính tổng i32:", "-n", "48", "--no-display-prompt", *cli_st],
    stdin=subprocess.DEVNULL,
    capture_output=True,
    text=True,
    check=True,
)
print("llama-cli smoke output:\n", smoke.stdout[:400])

## 6. Đẩy Checkpoint & GGUF lên Hugging Face Private Repo

In [ ]:
if PUSH_TO_HF:
    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(repo_id=TARGET_HF_REPO, private=True, exist_ok=True)
    api.upload_folder(
        repo_id=TARGET_HF_REPO,
        folder_path=str(FINAL_MERGED_DIR),
        commit_message="Upload Step 2 Stage 2A/2B merged BF16 Gemma 4 E4B checkpoint",
    )
    api.upload_folder(
        repo_id=TARGET_HF_REPO,
        folder_path=str(GGUF_DIR),
        path_in_repo="gguf",
        commit_message="Upload Step 2 modular Q4_K_M / IQ4_XS + mmproj GGUF artifacts",
    )
    print(f"Uploaded artifacts -> https://huggingface.co/{TARGET_HF_REPO}")
else:
    print("PUSH_TO_HF=False -> skipped upload.")